# 🔧 Data Preprocessing Pipeline

## 📋 Notebook Overview
This notebook performs comprehensive data preprocessing, applying One-Hot Encoding to categorical features and Label Encoding only to the target variable. This is the second step in our refactored ML pipeline.

### 🎯 Objectives:
- Load EDA results and insights from previous stage
- Apply One-Hot Encoding to all categorical features
- Apply Label Encoding only to the target variable
- Use Cramér's V for categorical-target relationship analysis
- Feature scaling and normalization for numerical variables
- Data quality checks and validation
- Prepare clean dataset for feature selection

### 📁 Input/Output:
- **Input**: `../data/Obesity_data.csv`, `../results/eda/eda_summary.json`
- **Output**: `../data/preprocessed_data.csv`, `../results/preprocessing/`

### 🔗 Refactored Workflow Position:
- **Previous**: `00_EDA.ipynb` → Exploratory Data Analysis  
- **Current**: `01_Preprocessing.ipynb` → Data Preprocessing (One-Hot + Label Encoding)
- **Next**: `02_Feature_Selection.ipynb` → Feature Engineering & Selection

---

## ⚠️ Original Content Preserved Below
*The original preprocessing logic is kept intact below, with new One-Hot Encoding approach added*

# 🔄 Data Preprocessing Pipeline (Refactored Workflow)

## ? Notebook Overview
This notebook performs comprehensive data preprocessing after EDA analysis. The refactored approach implements One-Hot encoding for categorical features and Label encoding only for the target variable, while preserving all original preprocessing methods as reference.

### 🎯 Objectives:
- Load raw obesity dataset and apply preprocessing transformations
- Implement One-Hot encoding for categorical features (NEW APPROACH)
- Apply Label encoding only for target variable (NObeyesdad)
- Handle missing values and outliers based on EDA insights
- Scale numerical features for algorithm compatibility
- Save preprocessed dataset for feature selection pipeline

### 📁 Input/Output:
- **Input**: `../data/Obesity_data.csv`, EDA analysis results
- **Output**: `../data/preprocessed_data.csv`, `../results/preprocessing/`

### 🔗 Refactored Workflow Position:
- **Previous**: `00_EDA.ipynb` → Exploratory Data Analysis & Cramér's V
- **Current**: `01_Preprocessing.ipynb` → Data Preprocessing (One-Hot + Label)
- **Next**: `02_Feature_Selection.ipynb` → Feature Engineering & Selection

### 🔄 Key Changes in Refactored Version:
- **Encoding Strategy**: One-Hot for categoricals, Label encoding ONLY for target
- **Statistical Analysis**: Incorporates Cramér's V insights from EDA
- **Workflow Integration**: Optimized for downstream feature selection
- **Original Methods**: All preserved as commented reference code

---

In [1]:
# Import libraries for preprocessing and EDA
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# Preprocessing tools
from sklearn.preprocessing import LabelEncoder, StandardScaler, MinMaxScaler
from sklearn.model_selection import train_test_split

# Statistical analysis
from scipy import stats
from scipy.stats import chi2_contingency

# Utilities
from pathlib import Path
import json
import warnings

warnings.filterwarnings('ignore')

# Set plotting style
plt.style.use('seaborn-v0_8')
sns.set_palette('husl')

print('✅ Libraries imported successfully!')

✅ Libraries imported successfully!


## 📂 Load Data from Previous Stage

In [2]:
# Load the dataset from data loading stage
data_path = Path('../data/loaded_data.csv')
target_info_path = Path('../data/target_classes.json')

# Ensure required directories exist
Path('../results').mkdir(exist_ok=True)

if data_path.exists():
    df = pd.read_csv(data_path)
    
    # Drop 'id' column if it exists (not useful for ML)
    if 'id' in df.columns:
        df = df.drop('id', axis=1)
        print('🗑️ Dropped \'id\' column (not needed for analysis)')
    
    print(f'✅ Dataset loaded successfully!')
    print(f'📊 Dataset shape: {df.shape}')
    
    # Load target information
    if target_info_path.exists():
        with open(target_info_path, 'r') as f:
            target_info = json.load(f)
        target_col = target_info['target_column']
        print(f'🎯 Target variable: {target_col}')
        print(f'🏷️ Target classes: {len(target_info["target_classes"])}')
    else:
        print('⚠️ Target info not found, using default target')
        target_col = 'NObeyesdad'
        target_info = {
            'target_column': target_col,
            'target_classes': list(df[target_col].unique()) if target_col in df.columns else [],
            'num_classes': len(df[target_col].unique()) if target_col in df.columns else 0
        }
else:
    print(f'❌ Dataset not found at {data_path}')
    print('Please run 01_Data_Loading.ipynb first')
    raise FileNotFoundError('Required data file not found')

🗑️ Dropped 'id' column (not needed for analysis)
✅ Dataset loaded successfully!
📊 Dataset shape: (20758, 17)
🎯 Target variable: NObeyesdad
🏷️ Target classes: 7


## 🔍 Advanced Exploratory Data Analysis

In [3]:
# Analyze feature types and their characteristics
numerical_features = df.select_dtypes(include=[np.number]).columns.tolist()
categorical_features = df.select_dtypes(include=['object']).columns.tolist()

# Remove target from categorical features list if present
if target_col in categorical_features:
    categorical_features.remove(target_col)

print('📊 FEATURE TYPE ANALYSIS')
print('=' * 60)
print(f'📈 Numerical features ({len(numerical_features)}):')
for i, feature in enumerate(numerical_features, 1):
    print(f'   {i:2d}. {feature}')

print(f'\n🏷️ Categorical features ({len(categorical_features)}):')
for i, feature in enumerate(categorical_features, 1):
    print(f'   {i:2d}. {feature}')

print(f'\n🎯 Target variable: {target_col}')
print(f'📊 Total features for analysis: {len(numerical_features) + len(categorical_features)}')

📊 FEATURE TYPE ANALYSIS
📈 Numerical features (8):
    1. Age
    2. Height
    3. Weight
    4. FCVC
    5. NCP
    6. CH2O
    7. FAF
    8. TUE

🏷️ Categorical features (8):
    1. Gender
    2. family_history_with_overweight
    3. FAVC
    4. CAEC
    5. SMOKE
    6. SCC
    7. CALC
    8. MTRANS

🎯 Target variable: NObeyesdad
📊 Total features for analysis: 16


In [4]:
# Initialize preprocessing variables and create working copy
df_processed = df.copy()
encoding_info = {}
label_encoders = {}

print('🔧 DATA PREPROCESSING INITIALIZATION')
print('=' * 60)
print(f'📊 Working with dataset shape: {df_processed.shape}')
print(f'🎯 Target variable: {target_col}')
print(f'🏷️ Categorical features to encode: {len(categorical_features)}')
print(f'📈 Numerical features to scale: {len(numerical_features)}')
print('\n🚀 Starting preprocessing pipeline...')

🔧 DATA PREPROCESSING INITIALIZATION
📊 Working with dataset shape: (20758, 17)
🎯 Target variable: NObeyesdad
🏷️ Categorical features to encode: 8
📈 Numerical features to scale: 8

🚀 Starting preprocessing pipeline...


In [5]:
# =============================================================================
# REFACTORED ENCODING APPROACH: One-Hot Encoding for Categorical Features
# =============================================================================

from sklearn.preprocessing import OneHotEncoder

print('🏷️ REFACTORED CATEGORICAL VARIABLE ENCODING')
print('=' * 60)
print('📋 New Approach: One-Hot Encoding for categorical features, Label Encoding only for target')
print()

# Apply Label Encoding ONLY to target variable
target_encoded_col = None
if target_col in df_processed.columns:
    le_target = LabelEncoder()
    target_encoded_col = f'{target_col}_encoded'
    df_processed[target_encoded_col] = le_target.fit_transform(df_processed[target_col])
    label_encoders['target'] = le_target
    
    # Store encoding mapping
    encoding_info[target_col] = dict(zip(
        le_target.classes_, 
        le_target.transform(le_target.classes_)
    ))
    
    print(f'🎯 Target \'{target_col}\' encoded with Label Encoding:')
    for original, encoded in encoding_info[target_col].items():
        print(f'   {original:<25} → {encoded}')
    print()

# Apply One-Hot Encoding to all categorical features
onehot_encoded_features = []
if len(categorical_features) > 0:
    print(f'🔧 Applying One-Hot Encoding to {len(categorical_features)} categorical features:')
    
    for i, feature in enumerate(categorical_features, 1):
        print(f'   {i}. 🏷️ One-Hot Encoding \'{feature}\'...')
        
        # Create dummy variables
        feature_dummies = pd.get_dummies(df_processed[feature], prefix=feature, prefix_sep='_')
        
        # Add dummy columns to dataframe
        df_processed = pd.concat([df_processed, feature_dummies], axis=1)
        
        # Track new column names
        new_columns = list(feature_dummies.columns)
        onehot_encoded_features.extend(new_columns)
        
        # Store encoding info for One-Hot
        encoding_info[f'{feature}_onehot'] = {
            'original_feature': feature,
            'encoded_columns': new_columns,
            'encoding_type': 'one_hot',
            'n_categories': len(new_columns)
        }
        
        print(f'      Created {len(new_columns)} binary features: {new_columns[:3]}{"..." if len(new_columns) > 3 else ""}')
    
    print(f'\n✅ Successfully applied One-Hot Encoding to {len(categorical_features)} categorical features')
    print(f'📊 Created {len(onehot_encoded_features)} new binary features')
else:
    print('📊 No categorical features found to encode')

print(f'\n📈 Dataset shape after encoding: {df_processed.shape}')



🏷️ REFACTORED CATEGORICAL VARIABLE ENCODING
📋 New Approach: One-Hot Encoding for categorical features, Label Encoding only for target

🎯 Target 'NObeyesdad' encoded with Label Encoding:
   Insufficient_Weight       → 0
   Normal_Weight             → 1
   Obesity_Type_I            → 2
   Obesity_Type_II           → 3
   Obesity_Type_III          → 4
   Overweight_Level_I        → 5
   Overweight_Level_II       → 6

🔧 Applying One-Hot Encoding to 8 categorical features:
   1. 🏷️ One-Hot Encoding 'Gender'...
      Created 2 binary features: ['Gender_Female', 'Gender_Male']
   2. 🏷️ One-Hot Encoding 'family_history_with_overweight'...
      Created 2 binary features: ['family_history_with_overweight_no', 'family_history_with_overweight_yes']
   3. 🏷️ One-Hot Encoding 'FAVC'...
      Created 2 binary features: ['FAVC_no', 'FAVC_yes']
   4. 🏷️ One-Hot Encoding 'CAEC'...
      Created 4 binary features: ['CAEC_Always', 'CAEC_Frequently', 'CAEC_Sometimes']...
   5. 🏷️ One-Hot Encoding 'SMOKE'.

In [6]:
# Scale numerical features
scaling_info = {}

if len(numerical_features) > 0:
    print('📏 NUMERICAL FEATURE SCALING')
    print('=' * 60)
    
    scaled_features = []
    print(f'🔧 Applying StandardScaler to {len(numerical_features)} numerical features:')
    
    for i, feature in enumerate(numerical_features, 1):
        scaled_name = f'{feature}_scaled'
        
        # Apply scaling
        scaler = StandardScaler()
        feature_data = df_processed[[feature]].values
        scaled_data = scaler.fit_transform(feature_data)
        df_processed[scaled_name] = scaled_data.flatten()
        scaled_features.append(scaled_name)
        
        # Store scaling statistics
        scaling_info[feature] = {
            'original_mean': float(df[feature].mean()),
            'original_std': float(df[feature].std()),
            'original_min': float(df[feature].min()),
            'original_max': float(df[feature].max()),
            'scaled_mean': float(df_processed[scaled_name].mean()),
            'scaled_std': float(df_processed[scaled_name].std()),
            'scaled_min': float(df_processed[scaled_name].min()),
            'scaled_max': float(df_processed[scaled_name].max())
        }
        
        print(f'\n   {i}. 📈 {feature} → {scaled_name}')
        print(f'      Original: μ={scaling_info[feature]["original_mean"]:.3f}, σ={scaling_info[feature]["original_std"]:.3f}')
        print(f'      Scaled: μ={scaling_info[feature]["scaled_mean"]:.3f}, σ={scaling_info[feature]["scaled_std"]:.3f}')
    
    print(f'\n✅ Successfully scaled {len(numerical_features)} numerical features')
else:
    print('📊 No numerical features found to scale')
    scaled_features = []

📏 NUMERICAL FEATURE SCALING
🔧 Applying StandardScaler to 8 numerical features:

   1. 📈 Age → Age_scaled
      Original: μ=23.842, σ=5.688
      Scaled: μ=0.000, σ=1.000

   2. 📈 Height → Height_scaled
      Original: μ=1.700, σ=0.087
      Scaled: μ=-0.000, σ=1.000

   3. 📈 Weight → Weight_scaled
      Original: μ=87.888, σ=26.379
      Scaled: μ=-0.000, σ=1.000

   4. 📈 FCVC → FCVC_scaled
      Original: μ=2.446, σ=0.533
      Scaled: μ=-0.000, σ=1.000

   5. 📈 NCP → NCP_scaled
      Original: μ=2.761, σ=0.705
      Scaled: μ=0.000, σ=1.000

   6. 📈 CH2O → CH2O_scaled
      Original: μ=2.029, σ=0.608
      Scaled: μ=-0.000, σ=1.000

   7. 📈 FAF → FAF_scaled
      Original: μ=0.982, σ=0.838
      Scaled: μ=0.000, σ=1.000

   8. 📈 TUE → TUE_scaled
      Original: μ=0.617, σ=0.602
      Scaled: μ=-0.000, σ=1.000

✅ Successfully scaled 8 numerical features


In [7]:
# Save processed dataset and all metadata
print('💾 SAVING PROCESSED DATA AND METADATA')
print('=' * 60)

# Ensure directories exist
Path('../data').mkdir(exist_ok=True)
Path('../results').mkdir(exist_ok=True)
Path('../results/preprocessing').mkdir(exist_ok=True)

# Save processed dataset
output_path = Path('../data/preprocessed_data.csv')
df_processed.to_csv(output_path, index=False)
print(f'✅ Processed data saved to: {output_path}')
print(f'📊 Final dataset shape: {df_processed.shape}')

# Create comprehensive preprocessing information
preprocessing_info = {
    'dataset_info': {
        'original_shape': list(df.shape),
        'processed_shape': list(df_processed.shape),
        'features_added': df_processed.shape[1] - df.shape[1]
    },
    'feature_types': {
        'original_numerical_features': numerical_features,
        'original_categorical_features': categorical_features,
        'scaled_features': scaled_features,
        'onehot_encoded_features': onehot_encoded_features if 'onehot_encoded_features' in locals() else [],
        'target_encoded_column': target_encoded_col
    },
    'encoding_approach': {
        'categorical_encoding': 'one_hot',  # Updated approach
        'target_encoding': 'label_encoding',
        'numerical_scaling': 'standard_scaler'
    },
    'target_info': {
        'target_column': target_col,
        'target_encoded_column': target_encoded_col,
        'target_classes': target_info.get('target_classes', []) if 'target_info' in locals() else list(df[target_col].unique()) if target_col in df.columns else [],
        'num_classes': len(df[target_col].unique()) if target_col in df.columns else 0
    },
    'preprocessing_applied': {
        'categorical_encoding': len(categorical_features) > 0,
        'numerical_scaling': len(numerical_features) > 0,
        'one_hot_encoding': 'onehot_encoded_features' in locals() and len(onehot_encoded_features) > 0
    },
    'quality_metrics': {
        'missing_values': int(df_processed.isnull().sum().sum()),
        'duplicate_rows': int(df_processed.duplicated().sum())
    },
    'processing_timestamp': pd.Timestamp.now().isoformat()
}

# Save preprocessing info
preprocessing_path = Path('../results/preprocessing/preprocessing_info.json')
with open(preprocessing_path, 'w') as f:
    json.dump(preprocessing_info, f, indent=2)
print(f'📋 Preprocessing info saved to: {preprocessing_path}')

# Fix encoding info to use native Python types and handle One-Hot encoding structure
encoding_info_fixed = {}
for feature, mapping in encoding_info.items():
    if isinstance(mapping, dict):
        if 'encoded_columns' in mapping:  # One-Hot encoding info
            encoding_info_fixed[feature] = mapping
        else:  # Label encoding info
            encoding_info_fixed[feature] = {str(k): int(v) for k, v in mapping.items()}
    else:
        encoding_info_fixed[feature] = mapping

# Save encoding mappings
encoding_path = Path('../results/preprocessing/encoding_mappings.json')
with open(encoding_path, 'w') as f:
    json.dump(encoding_info_fixed, f, indent=2)
print(f'🗂️ Encoding mappings saved to: {encoding_path}')

# Save scaling information
if scaling_info:
    scaling_path = Path('../results/preprocessing/scaling_info.json')
    with open(scaling_path, 'w') as f:
        json.dump(scaling_info, f, indent=2)
    print(f'📏 Scaling info saved to: {scaling_path}')

# Save feature names for easy reference
feature_info = {
    'original_features': list(df.columns),
    'preprocessed_features': list(df_processed.columns),
    'numerical_features': numerical_features,
    'original_categorical_features': categorical_features,
    'onehot_encoded_features': onehot_encoded_features if 'onehot_encoded_features' in locals() else [],
    'scaled_features': scaled_features,
    'target_column': target_col,
    'target_encoded_column': target_encoded_col
}

feature_info_path = Path('../results/preprocessing/feature_info.json')
with open(feature_info_path, 'w') as f:
    json.dump(feature_info, f, indent=2)
print(f'🔍 Feature information saved to: {feature_info_path}')

print(f'\n📊 REFACTORED PREPROCESSING SUMMARY:')
print(f'   Original features: {len(df.columns)}')
print(f'   Processed features: {len(df_processed.columns)}')
print(f'   Features added: {df_processed.shape[1] - df.shape[1]}')
print(f'   Categorical features (One-Hot): {len(categorical_features)}')
print(f'   One-Hot encoded columns: {len(onehot_encoded_features) if "onehot_encoded_features" in locals() else 0}')
print(f'   Numerical scaled: {len(numerical_features)}')
print(f'   Target encoded (Label): {"Yes" if target_encoded_col else "No"}')
print(f'\n🎯 READY FOR NEXT STAGE: Feature Selection & Engineering')
print(f'🔄 Next notebook: 02_Feature_Selection.ipynb')

💾 SAVING PROCESSED DATA AND METADATA
✅ Processed data saved to: ..\data\preprocessed_data.csv
📊 Final dataset shape: (20758, 48)
📋 Preprocessing info saved to: ..\results\preprocessing\preprocessing_info.json
🗂️ Encoding mappings saved to: ..\results\preprocessing\encoding_mappings.json
📏 Scaling info saved to: ..\results\preprocessing\scaling_info.json
🔍 Feature information saved to: ..\results\preprocessing\feature_info.json

📊 REFACTORED PREPROCESSING SUMMARY:
   Original features: 17
   Processed features: 48
   Features added: 31
   Categorical features (One-Hot): 8
   One-Hot encoded columns: 22
   Numerical scaled: 8
   Target encoded (Label): Yes

🎯 READY FOR NEXT STAGE: Feature Selection & Engineering
🔄 Next notebook: 02_Feature_Selection.ipynb
✅ Processed data saved to: ..\data\preprocessed_data.csv
📊 Final dataset shape: (20758, 48)
📋 Preprocessing info saved to: ..\results\preprocessing\preprocessing_info.json
🗂️ Encoding mappings saved to: ..\results\preprocessing\encoding